In [2]:
%pip install pandas
%pip install gdown
%pip install numpy
%pip install pyarrow
%pip install polars
%pip install statsmodels


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[noti

In [3]:
import pandas as pd

df = pd.read_parquet("final_set.parquet")

import os
import gdown

FILE_ID = "1ACJpnOYLd8QKy9GYqPRaO3l157vZGkQA"
OUTPUT = "FINAL1000Stocks.parquet"

if not os.path.exists(OUTPUT):
    print("Downloading FINAL1000Stocks.parquet...")
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, OUTPUT, quiet=False)
else:
    print("File already exists. Skipping download.")









File already exists. Skipping download.


In [3]:
import polars as pl

df = pl.read_parquet("FINAL1000Stocks.parquet")

print(df)

: 

In [3]:
import pyarrow.parquet as pq
import pandas as pd

df = pd.read_parquet("FINAL1000Stocks.parquet", engine="pyarrow")

print(df.info())

: 

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

INPUT_FILE = Path("final_set.parquet")
OUTPUT_FILE = Path("final_set_fracdiff.parquet")

# -----------------------------
# Load raw file
# -----------------------------
df = pd.read_parquet(INPUT_FILE)
print("Original shape:", df.shape)


# -----------------------------
# Cleaning Function
# -----------------------------
def clean_data(df_symbol):
    df_symbol = df_symbol.copy()

    # Ensure datetime
    df_symbol["date"] = pd.to_datetime(df_symbol["date"], errors="coerce")
    df_symbol = df_symbol.sort_values("date").set_index("date")

    # Remove duplicates
    df_symbol = df_symbol[~df_symbol.index.duplicated(keep="first")]

    # Force numeric close
    df_symbol["close"] = pd.to_numeric(df_symbol["close"], errors="coerce")

    # Remove NaNs and non-positive
    df_symbol = df_symbol.dropna(subset=["close"])
    df_symbol = df_symbol[df_symbol["close"] > 0]

    # Winsorize extremes
    q_low = df_symbol["close"].quantile(0.001)
    q_high = df_symbol["close"].quantile(0.999)
    df_symbol["close"] = df_symbol["close"].clip(q_low, q_high)

    # Ensure enough history
    if len(df_symbol) < 100:
        return None

    return df_symbol


# -----------------------------
# Fixed-Width Fractional Weights
# -----------------------------
def get_weights_ffd(d, thresh=1e-5):
    """
    Generate fractional differentiation weights
    using fixed-width window approach.
    """
    w = [1.0]
    k = 1

    while True:
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < thresh:
            break
        w.append(w_k)
        k += 1

    return np.array(w[::-1])  # reverse for convolution order


# -----------------------------
# Expanding-Window FFD (No NaNs)
# -----------------------------
def frac_diff_ffd_full(series, d, thresh=1e-5):
    """
    Fractional differentiation using expanding window
    so that ALL rows receive a value (no NaNs).
    """
    weights = get_weights_ffd(d, thresh)
    width = len(weights)

    output = pd.Series(index=series.index, dtype="float64")

    for i in range(len(series)):
        start = max(0, i - width + 1)
        window = series.iloc[start:i + 1]

        if window.isnull().any():
            continue

        # Match weights to available window length
        w = weights[-len(window):]

        output.iloc[i] = float(np.dot(w, window))

    return output


# -----------------------------
# Main Pipeline
# -----------------------------
D = 0.5
results = []

for ticker, df_symbol in df.groupby("ticker"):

    cleaned = clean_data(df_symbol)
    if cleaned is None:
        continue

    fd_series = frac_diff_ffd_full(cleaned["close"], D)

    final_symbol = cleaned.copy()
    final_symbol["frac_diff"] = fd_series
    final_symbol["ticker"] = ticker

    results.append(final_symbol.reset_index())


# -----------------------------
# Combine and Save
# -----------------------------
if results:
    final_df = pd.concat(results, axis=0, ignore_index=True)
    final_df = final_df.sort_values(["ticker", "date"])
    final_df.to_parquet(OUTPUT_FILE)

    print("Saved cleaned file with FFD frac_diff to:", OUTPUT_FILE.resolve())
    print("Final shape:", final_df.shape)
    print("Total NaNs in frac_diff:", final_df["frac_diff"].isna().sum())
else:
    print("No valid tickers processed.")

print(final_df)

Original shape: (1232, 7)
Saved cleaned file with FFD frac_diff to: /workspaces/CSC491-Project/final_set_fracdiff.parquet
Final shape: (1025, 8)
Total NaNs in frac_diff: 0
           date ticker      open      high       low     close  dollar volume  \
0    2025-04-21   MSFT  363.1500  363.5100  363.1500  364.2954   1.387097e+08   
1    2025-04-22   MSFT  361.9092  367.6430  360.0100  367.3600   1.280425e+08   
2    2025-04-23   MSFT  375.5000  380.2400  373.1900  374.5000   1.472605e+08   
3    2025-04-24   MSFT  375.5300  388.0500  375.5300  386.5400   1.623831e+08   
4    2025-04-25   MSFT  388.8150  391.9700  384.7000  391.8500   1.414150e+08   
...         ...    ...       ...       ...       ...       ...            ...   
1020 2026-02-05   TSLA  397.1700  401.8600  387.9200  397.2500   5.280360e+08   
1021 2026-02-06   TSLA  398.7300  414.5270  398.7300  411.0426   4.837810e+08   
1022 2026-02-09   TSLA  409.9703  421.1612  408.0306  417.2000   4.325091e+08   
1023 2026-02-10   